# INSIDER ALPHA — Real Multi-Agent System

## Single Market Deep Dive

This notebook demonstrates the **real agents** from the polymarket code repository analyzing ONE prediction market.

**What you'll see:**
- Agent A: Real insider risk analysis (LLM call)
- Agent B: Real market behavior analysis (LLM + price tools)
- Revision Agent: Real pattern detection & QA
- Decision Agent: Real weighted investment decision

**Duration:** ~3 minutes (1 LLM call per agent × 4 = 4 API calls)

---

## 1. Setup: Paths and Imports

In [29]:
import sys
from pathlib import Path
import os
from dotenv import load_dotenv
from datetime import datetime, timezone
import json

# Setup paths — works whether the notebook is run from:
#   (a) the project root  (polymarket-events/)
#   (b) a sibling demo/   folder

current_file = Path.cwd()

# Locate demo_dir (where data/ lives)
if (current_file / "demo" / "data").exists():
    # Running from project root — demo data is in ./demo/data/
    demo_dir = current_file / "demo"
elif (current_file / "data" / "exports").exists():
    # Running from inside the demo/ folder directly
    demo_dir = current_file
else:
    demo_dir = current_file / "demo"  # fallback

# Project dir = parent of demo_dir
project_dir = demo_dir.parent

# Locate the code folder (has src/ai_layer/) — check project_dir itself first,
# then its sub-directories (handles both single-repo and nested structures)
candidate_dirs = []

# Case A: notebook is at project root -> src/ lives directly in project_dir
if (project_dir / "src" / "ai_layer").exists():
    candidate_dirs.append(project_dir)

# Case B: project_dir has child repos (classic demo-alongside-repo layout)
for item in project_dir.iterdir():
    if item.is_dir() and (item / "src" / "ai_layer").exists():
        if item not in candidate_dirs:
            candidate_dirs.append(item)

if not candidate_dirs:
    raise FileNotFoundError(
        "\nERROR: Cannot find the polymarket code folder!\n"
        f"Expected a folder with 'src/ai_layer/' inside: {project_dir}\n"
        "Make sure you have cloned the polymarket repository.\n"
    )

if len(candidate_dirs) > 1:
    remove_pandas = [d for d in candidate_dirs if "remove-pandas" in d.name]
    master_dir = remove_pandas[0] if remove_pandas else max(candidate_dirs, key=lambda p: p.stat().st_mtime)
else:
    master_dir = candidate_dirs[0]

print(f"Demo dir:      {demo_dir}")
print(f"Project dir:   {project_dir}")
print(f"Master dir:    {master_dir}")

parquet_exists = (demo_dir / "data" / "exports" / "polymarket_tagged_sample.parquet").exists()
print(f"\nDemo data found: {parquet_exists}")
print(f"Master dir exists: {master_dir.exists()}")

# Add to Python path
sys.path.insert(0, str(master_dir))
sys.path.insert(0, str(project_dir))

# Load .env — check demo_dir first, then project root
env_path = demo_dir / ".env"
if not env_path.exists():
    env_path = project_dir / ".env"
load_dotenv(env_path)
print(f"\n.env loaded from: {env_path}")

has_key = bool(os.getenv("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY configured: {has_key}")

if not has_key:
    env_target = project_dir / ".env"
    raise ValueError(
        "\nERROR: OPENAI_API_KEY not found!\n"
        f"Create a .env file at {env_target} with:\n"
        "  OPENAI_API_KEY=sk-proj-YOUR_KEY_HERE\n"
    )


Demo dir:      c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects\Polymarket Events\demo
Project dir:   c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects\Polymarket Events
Master dir:    c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects\Polymarket Events

Demo data found: True
Master dir exists: True

.env loaded from: c:\Users\jsilvaec\OneDrive - The University of Chicago\Desktop\Personal Projects\Polymarket Events\.env
OPENAI_API_KEY configured: True


---

## 2. Import Real Agents from Master 2

In [30]:
# Import real agents
from src.ai_layer.agent_a.agent import agent_a_initial
from src.ai_layer.agent_a.schemas import AgentAInputPackage
from src.ai_layer.agent_a.params import AgentAParams

from src.ai_layer.agent_b.agent import agent_b_initial
from src.ai_layer.agent_b.schemas import AgentBInputPackage
from src.ai_layer.agent_b.params import AgentBParams

from src.ai_layer.revision_agent import revision_agent, revision_agent_deterministic

from src.ai_layer.decision_agent.agent import decision_agent, decision_agent_deterministic
from src.ai_layer.decision_agent.schemas import DecisionAgentInputPackage
from src.ai_layer.decision_agent.params import DecisionAgentParams

print("All agents imported successfully (LLM + deterministic versions)")

All agents imported successfully (LLM + deterministic versions)


---

## 3. Load Real Market Data

In [ ]:
import pandas as pd
import sqlite3
from datetime import datetime, timezone, timedelta
import ipywidgets as widgets
from IPython.display import display

# Load real polymarket data
parquet_path = demo_dir / "data/exports/polymarket_tagged_sample.parquet"
df_raw = pd.read_parquet(parquet_path)

print(f"Loaded {len(df_raw):,} markets from parquet")
print(f"Columns: {list(df_raw.columns)[:8]}...\n")

# Load price history database
db_path = demo_dir / "data/price_history.db"
conn = sqlite3.connect(db_path)
price_df = pd.read_sql_query("SELECT market_id, timestamp, price FROM price_history", conn)
conn.close()

print(f"Loaded {len(price_df):,} price points from price_history.db\n")

# Filter markets
df_filtered = df_raw[
    (df_raw['volume'] >= 20_000) &
    (df_raw['status'] == 'closed') &
    (df_raw['end_date'].notna())
].copy()

# Remove boring categories
boring = ['Crypto', 'Elections', 'Sports', 'NBA', 'NFL']
for cat in boring:
    df_filtered = df_filtered[~df_filtered['category'].str.contains(cat, case=False, na=False)]

# Sort by volume (descending) to get highest-volume markets first
df_filtered = df_filtered.sort_values('volume', ascending=False).reset_index(drop=True)

# Find default index for GPT-4.5 market
default_idx = 0
for idx, row in df_filtered.iterrows():
    if str(row['market_id']) == "523151":
        default_idx = idx
        break

# --- Interactive slider with HTML preview (no duplication) ---
market_slider = widgets.IntSlider(
    value=default_idx, min=0, max=len(df_filtered) - 1, step=1,
    description="Market #",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "70px"},
    continuous_update=False,
)

preview_label = widgets.HTML(value="")

def _update_preview(change):
    row = df_filtered.iloc[change["new"]]
    vol = row["volume"]
    vol_str = f"${vol/1_000_000:.1f}M" if vol >= 1_000_000 else f"${vol/1_000:.0f}k"
    mid = row["market_id"]
    n_prices = len(price_df[price_df["market_id"] == str(int(mid))])
    preview_label.value = (
        f"<div style='padding: 6px 0; font-family: monospace;'>"
        f"<b>{row['question']}</b><br>"
        f"Volume: {vol_str} &nbsp;|&nbsp; Price points: {n_prices} &nbsp;|&nbsp; ID: {mid}"
        f"</div>"
    )

market_slider.observe(_update_preview, names="value")

print("Select a market to analyze (drag slider, then re-run cells below):\n")
display(market_slider)
display(preview_label)

# Trigger initial preview
_update_preview({"new": market_slider.value})

In [32]:
# Use the selected market (change slider above and re-run from here)
market = df_filtered.iloc[market_slider.value]

print("-" * 100)
print(f"\n🎯 ANALYZING MARKET:")
print(f"\nMarket ID: {market['market_id']}")
print(f"Question: {market['question']}")
print(f"Category: {market['category']}")
print(f"Volume: ${market['volume']:,.0f}")
print(f"Status: {market['status']}")
print(f"End Date: {market['end_date']}")

# Get price history for this market
market_id_str = str(int(market['market_id']))
market_prices = price_df[price_df['market_id'] == market_id_str].copy()
print(f"\n💰 Price history points available: {len(market_prices)}")
if len(market_prices) > 0:
    print(f"   Price range: ${market_prices['price'].min():.3f} - ${market_prices['price'].max():.3f}")


----------------------------------------------------------------------------------------------------

🎯 ANALYZING MARKET:

Market ID: 523151
Question: Will GPT-4.5 be released by February 28?
Category: 
Volume: $141,504
Status: closed
End Date: 2025-02-28 12:00:00+00:00

💰 Price history points available: 32
   Price range: $0.105 - $0.996


---

## 4. AGENT A: Insider Risk Assessment

Real LLM call to analyze insider risk from market description.

In [33]:
import warnings
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

print("🔍 AGENT A: Insider Risk Assessment\n")
print(f"Input: Market title + description (blind to price/volume)\n")

# Create Agent A input
pkg_a = AgentAInputPackage(
    market_id=str(market['market_id']),
    question=market['question'],
    description=market['question'],  # description not always available
    category=market.get('category', ''),
    tags=market.get('tags', '').split('|')[:5] if isinstance(market.get('tags'), str) else [],
    platform="polymarket",
    end_date=pd.Timestamp(market['end_date']).to_pydatetime().replace(tzinfo=timezone.utc) if pd.notna(market['end_date']) else datetime.now(timezone.utc),
)

# Run Agent A (REAL LLM call — caching disabled for demo)
params_a = AgentAParams()
params_a.cache_enabled = False
report_a = agent_a_initial(pkg_a, params_a)

print(f"✅ Agent A Complete\n")
print(f"Insider Risk Score: {report_a.insider_risk_score}/10")
print(f"Confidence: {report_a.confidence}")
print(f"\nReasoning:\n  {report_a.reasoning}\n")
print(f"Info Holders: {', '.join(report_a.info_holders[:5])}")
print(f"Leak Vectors: {', '.join(report_a.leak_vectors[:5])}")

🔍 AGENT A: Insider Risk Assessment

Input: Market title + description (blind to price/volume)

✅ Agent A Complete

Insider Risk Score: 7/10
Confidence: high

Reasoning:
  The release of GPT-4.5 is likely known to a small, identifiable group, including OpenAI executives and developers, who have clear advance knowledge of the timeline. The financial incentive is significant due to the potential for large price swings in the prediction market, and there are realistic leak pathways through social media or insider trading. While there may be some barriers to leakage, the potential rewards create a strong motivation for insiders to share information.

Info Holders: OpenAI executives, AI developers, project managers at OpenAI
Leak Vectors: social media posts by insiders, information shared with connected traders, leaks to tech journalists


---

## 5. AGENT B: Market Behavior Analysis

Real analysis of price/volume anomalies (real price tools).

In [34]:
print("🔍 AGENT B: Market Behavior Analysis\n")
print(f"Input: Price history (last 5 days) + volume data\n")

# Get price history for this market and convert to PricePoint objects
from src.data_layer.models import PricePoint

# Build price history list
price_list = []
if len(market_prices) > 0:
    market_prices_sorted = market_prices.sort_values('timestamp')
    for _, row in market_prices_sorted.iterrows():
        # Convert Unix timestamp to datetime
        ts = datetime.fromtimestamp(row['timestamp'], tz=timezone.utc)
        price = float(row['price'])
        price_list.append(PricePoint(timestamp=ts, price=price, raw_price=price))

# Match backtest: evaluation_date = end_date - 24h, window = [end_date-120h, end_date-24h]
end_dt = pd.Timestamp(market['end_date']).to_pydatetime().replace(tzinfo=timezone.utc) if pd.notna(market['end_date']) else datetime.now(timezone.utc)
eval_date = end_dt - timedelta(hours=24)
window_start = end_dt - timedelta(hours=120)

# Filter price history to backtest window
price_list_window = [p for p in price_list if window_start <= p.timestamp <= eval_date]

if len(price_list_window) > 0:
    print(f"✅ Using {len(price_list_window)} price points in [end-120h, end-24h] window")
    first_ts = price_list_window[0].timestamp
    last_ts = price_list_window[-1].timestamp
    print(f"   Time range: {first_ts.date()} to {last_ts.date()}")
    print(f"   Price range: ${min(p.price for p in price_list_window):.3f} - ${max(p.price for p in price_list_window):.3f}")
else:
    print(f"⚠️  No price data in backtest window, using all available ({len(price_list)} points)")
    price_list_window = price_list

# Current price = price at evaluation_date (last point in window), matching backtest
current_price = price_list_window[-1].price if price_list_window else (float(market['yes_price']) if pd.notna(market['yes_price']) else 0.5)

# Volume and market age, matching backtest
volume_raw = market.get('volume', None)
volume_total = float(volume_raw) if pd.notna(volume_raw) else None

market_age_days = None
if pd.notna(market.get('start_date')) and pd.notna(market.get('end_date')):
    start_dt = pd.Timestamp(market['start_date']).to_pydatetime().replace(tzinfo=timezone.utc)
    market_age_days = (end_dt - start_dt).total_seconds() / 86_400

# Create Agent B input matching backtest exactly
pkg_b = AgentBInputPackage(
    evaluation_date=eval_date,
    end_date=end_dt,
    price_history=price_list_window,
    current_price=current_price,
    volume_total_usd=volume_total,
    market_age_days=market_age_days,
)

# Run Agent B (REAL LLM call with backtest-matching inputs)
try:
    report_b = agent_b_initial(pkg_b, AgentBParams())
    print(f"\n✅ Agent B Complete\n")
    print(f"Behavior Score: {report_b.behavior_score}/10")
    print(f"Signal Direction: {report_b.signal_direction}")
    print(f"Confidence: {report_b.confidence}")
    print(f"\nReasoning:\n  {report_b.reasoning}")
    print(f"\nKey Findings:")
    for finding in report_b.key_findings:
        print(f"  • {finding}")
except Exception as e:
    print(f"⚠️  Agent B error: {str(e)}")

🔍 AGENT B: Market Behavior Analysis

Input: Price history (last 5 days) + volume data

✅ Using 8 price points in [end-120h, end-24h] window
   Time range: 2025-02-23 to 2025-02-27
   Price range: $0.125 - $0.825

✅ Agent B Complete

Behavior Score: 9/10
Signal Direction: YES
Confidence: high

Reasoning:
  The price jump of 70pp is significant and sustained, occurring within 24 hours of market close, which is a strong indicator of upward movement. The momentum analysis also confirms this direction, showing a consistent upward trend. Given the absence of volume data does not negate the strong price signal, the overall assessment is highly confident in recommending a YES direction.

Key Findings:
  • Detected a sustained price jump of 70pp from 0.125 to 0.825, indicating strong upward movement.
  • Momentum analysis shows a consistent upward trend with a dominant direction of UP, supporting the price jump.


---

## 6. Interactive Controls

Choose which method and parameters the Revision and Decision agents use.
Toggle between the **LLM-based** and **deterministic** implementations, and select
from the 6 backtest configurations for the Decision Agent.

In [35]:
import ipywidgets as widgets
from IPython.display import display

# --- Revision Agent controls ---
revision_method = widgets.RadioButtons(
    options=["Deterministic (instant, rule-based)", "LLM (real API call)"],
    value="Deterministic (instant, rule-based)",
    description="",
    layout=widgets.Layout(width="auto"),
)

# --- Decision Agent controls ---
DECISION_CONFIGS = {
    "default": dict(min_a_score=4, min_b_score=4, weight_a=0.4, weight_b=0.6,
                    go_score_threshold=6.0, min_edge_pp=5.0),
    "aggressive": dict(min_a_score=3, min_b_score=3, weight_a=0.4, weight_b=0.6,
                       go_score_threshold=5.0, min_edge_pp=3.0),
    "conservative": dict(min_a_score=5, min_b_score=5, weight_a=0.4, weight_b=0.6,
                         go_score_threshold=7.0, min_edge_pp=7.0),
    "a_heavy": dict(min_a_score=4, min_b_score=4, weight_a=0.6, weight_b=0.4,
                    go_score_threshold=6.0, min_edge_pp=5.0),
    "b_heavy": dict(min_a_score=4, min_b_score=4, weight_a=0.3, weight_b=0.7,
                    go_score_threshold=6.0, min_edge_pp=5.0),
    "no_conf_penalty": dict(min_a_score=4, min_b_score=4, weight_a=0.4, weight_b=0.6,
                            go_score_threshold=6.0, min_edge_pp=5.0,
                            confidence_high=1.0, confidence_medium=1.0, confidence_low=1.0),
}

decision_config = widgets.Dropdown(
    options=list(DECISION_CONFIGS.keys()),
    value="aggressive",
    description="",
    layout=widgets.Layout(width="200px"),
)

decision_method = widgets.RadioButtons(
    options=["Deterministic (instant, formula-based)", "LLM (real API call)"],
    value="Deterministic (instant, formula-based)",
    description="",
    layout=widgets.Layout(width="auto"),
)

# --- Layout ---
revision_box = widgets.VBox([
    widgets.HTML("<b>Revision Agent Method</b>"),
    revision_method,
])

decision_box = widgets.VBox([
    widgets.HTML("<b>Decision Agent Config</b>"),
    decision_config,
    widgets.HTML("<br><b>Decision Agent Method</b>"),
    decision_method,
])

controls = widgets.HBox([revision_box, widgets.HTML("&nbsp;" * 10), decision_box],
                         layout=widgets.Layout(justify_content="space-around"))
display(controls)

In [36]:
import time as _time

use_llm_revision = revision_method.value.startswith("LLM")
method_label = "LLM" if use_llm_revision else "Deterministic"

print(f"REVISION AGENT: Pattern Detection & QA  [{method_label}]\n")
print(f"Input: Agent A + Agent B reports\n")

try:
    _t0 = _time.perf_counter()

    if use_llm_revision:
        revision_out = revision_agent(
            agent_a_report=report_a.model_dump(),
            agent_b_report=report_b.model_dump() if 'report_b' in locals() else {},
        )
    else:
        revision_out = revision_agent_deterministic(
            agent_a_report=report_a.model_dump(),
            agent_b_report=report_b.model_dump() if 'report_b' in locals() else {},
        )

    _elapsed = _time.perf_counter() - _t0
    print(f"Revision Agent Complete  ({method_label}, {_elapsed:.3f}s)\n")
    print(f"Detected Pattern: {revision_out.revision_flag}")
    print(f"Recommendation to Decision Agent: {revision_out.recommendation_to_decision_agent}")

    # Show the best available explanation text
    explanation = (
        revision_out.flag_explanation
        or revision_out.revision_notes
        or (revision_out.llm_reasoning_summary if revision_out.llm_reasoning_summary else "")
    )
    if explanation.strip():
        print(f"\nExplanation:\n  {explanation}")

    if revision_out.feedback_to_send:
        print(f"\nFeedback generated:")
        for fb in revision_out.feedback_to_send:
            print(f"  -> Agent {fb.recipient}: {fb.message}")
        print(f"\n  Note: In the full LangGraph pipeline, this feedback would be routed back")
        print(f"  to Agent {revision_out.feedback_to_send[0].recipient} for re-analysis (up to 2 iterations). This demo runs")
        print(f"  each agent once — see src/graph.py for the full feedback loop.")

except Exception as e:
    print(f"Revision Agent error: {str(e)}")

REVISION AGENT: Pattern Detection & QA  [Deterministic]

Input: Agent A + Agent B reports

Revision Agent Complete  (Deterministic, 0.000s)

Detected Pattern: DIRECTIONAL_CONFLICT
Recommendation to Decision Agent: GO_EVALUATE

Explanation:
  Both agents signal high confidence (A=7, B=9) with B direction=YES. Decision Agent resolves via weighting.


---

## 7. DECISION AGENT: Final Investment Decision

Real Bayesian-weighted decision: GO or SKIP?

In [37]:
use_llm_decision = decision_method.value.startswith("LLM")
config_name = decision_config.value
config_overrides = DECISION_CONFIGS[config_name]
method_label = "LLM" if use_llm_decision else "Deterministic"

print(f"DECISION AGENT: Final Investment Decision  [{method_label}]\n")
print(f"Input: Agent A + Agent B + Revision reports\n")

# Only show config parameters for deterministic (LLM doesn't use them)
if not use_llm_decision:
    print(f"Parameters ({config_name}):")
    for k, v in config_overrides.items():
        print(f"  {k}: {v}")
    print()
else:
    print(f"(LLM mode — the model reasons freely; config parameters are not used)\n")

try:
    rev_rec = revision_out.recommendation_to_decision_agent if 'revision_out' in locals() else "GO_EVALUATE"

    pkg_dec = DecisionAgentInputPackage(
        revision_flag=revision_out.revision_flag if 'revision_out' in locals() else "NONE",
        flag_explanation=revision_out.flag_explanation if 'revision_out' in locals() else "",
        agent_a_report=report_a.model_dump(),
        agent_b_report=report_b.model_dump() if 'report_b' in locals() else {},
        revision_notes=revision_out.revision_notes if 'revision_out' in locals() else "",
        recommendation_to_decision_agent=rev_rec,
        current_market_price=current_price,
        evaluation_date=eval_date,
        end_date=end_dt,
        market_id=str(market['market_id']),
    )

    params = DecisionAgentParams(**config_overrides)

    _t0 = _time.perf_counter()

    if use_llm_decision:
        decision = decision_agent(pkg_dec, params)
    else:
        decision = decision_agent_deterministic(pkg_dec, params)

    _elapsed = _time.perf_counter() - _t0

    # Detect early exit (both LLM and deterministic skip the call if Revision said SKIP/WATCH)
    early_exit = rev_rec in ("SKIP", "WATCH")
    if early_exit:
        print(f"Decision Agent Complete  ({_elapsed:.3f}s — early exit, Revision Agent already decided {rev_rec})\n")
    else:
        print(f"Decision Agent Complete  ({method_label}, {_elapsed:.3f}s)\n")

    # --- Core decision ---
    action = decision.recommendation.get("action", "?") if decision.recommendation else "?"
    print(f"Action: {action}")
    if decision.bet_direction and decision.bet_direction != "null":
        print(f"Bet Direction: {decision.bet_direction}")
    print(f"Entry Price: ${current_price:.2f}")

    # --- Analysis (only show fields that have real values) ---
    a = decision.analysis or {}
    has_scores = a.get("weighted_score") is not None
    has_edge = a.get("estimated_edge_pp") is not None and a.get("estimated_edge_pp") != 0

    if has_scores or has_edge:
        print(f"\nAnalysis:")
        if a.get("agent_a_score") is not None:
            print(f"  Agent A Score: {a['agent_a_score']}/10")
        if a.get("agent_b_score") is not None:
            print(f"  Agent B Score: {a['agent_b_score']}/10")
        if has_scores:
            wa = a.get("weight_a_percentage")
            wb = a.get("weight_b_percentage")
            if wa is not None and wb is not None:
                print(f"  Weights: A={wa}%, B={wb}%")
            print(f"  Weighted Score: {a['weighted_score']}/10")
        if has_edge:
            print(f"  Estimated Edge: {a['estimated_edge_pp']:.1f}pp")
        edge_text = a.get("edge_assessment", "")
        if edge_text:
            print(f"  Edge Assessment: {edge_text}")

    # --- Full reasoning (LLM mode often has rich text here) ---
    if not early_exit and use_llm_decision and decision.full_reasoning:
        reasoning = decision.full_reasoning.strip()
        if reasoning:
            print(f"\nReasoning:\n  {reasoning}")

except Exception as e:
    print(f"Decision Agent error: {str(e)}")

DECISION AGENT: Final Investment Decision  [Deterministic]

Input: Agent A + Agent B + Revision reports

Parameters (aggressive):
  min_a_score: 3
  min_b_score: 3
  weight_a: 0.4
  weight_b: 0.6
  go_score_threshold: 5.0
  min_edge_pp: 3.0

Decision Agent Complete  (Deterministic, 0.000s)

Action: INVEST
Bet Direction: YES
Entry Price: $0.82

Analysis:
  Agent A Score: 7/10
  Agent B Score: 9/10
  Weights: A=40%, B=60%
  Weighted Score: 8.2/10
  Estimated Edge: 12.8pp
  Edge Assessment: meaningful


---

## 8. Summary: The Complete Pipeline

How the real system works end-to-end.

In [38]:
print("INSIDER ALPHA PIPELINE — REAL EXECUTION\n")
print("INPUT: 1 real prediction market")
print(f"  Question: {market['question'][:70]}...")
print(f"  Volume: ${market['volume']:,.0f}")
print(f"  Entry Price: ${current_price:.2f}\n")

print("AGENT A: Insider risk analysis (LLM call)")
print(f"  Score: {report_a.insider_risk_score}/10")
print(f"  Confidence: {report_a.confidence}\n")

print("AGENT B: Market behavior analysis (LLM + tools)")
if 'report_b' in locals():
    print(f"  Score: {report_b.behavior_score}/10")
    print(f"  Confidence: {report_b.confidence}")
    dir_label = {"YES": "price moving toward YES", "NO": "price moving toward NO", "SKIP": "no clear signal"}
    print(f"  Signal Direction: {report_b.signal_direction} ({dir_label.get(report_b.signal_direction, '')})\n")

rev_method = "LLM" if revision_method.value.startswith("LLM") else "Deterministic"
print(f"REVISION AGENT: Pattern detection ({rev_method})")
if 'revision_out' in locals():
    print(f"  Pattern: {revision_out.revision_flag}")
    print(f"  Recommendation: {revision_out.recommendation_to_decision_agent}")
    explanation = (
        revision_out.flag_explanation
        or revision_out.revision_notes
        or (revision_out.llm_reasoning_summary if revision_out.llm_reasoning_summary else "")
    )
    if explanation.strip():
        print(f"  Explanation: {explanation[:200]}")
    print()

dec_method = "LLM" if decision_method.value.startswith("LLM") else "Deterministic"
print(f"DECISION AGENT: Investment decision ({dec_method}, config={decision_config.value})")
if 'decision' in locals():
    action = decision.recommendation.get("action", "?") if decision.recommendation else "?"
    print(f"  Action: {action}")
    if decision.bet_direction and decision.bet_direction != "null":
        print(f"  Bet Direction: {decision.bet_direction}")
    a = decision.analysis or {}
    edge = a.get("estimated_edge_pp")
    if edge is not None and edge != 0:
        print(f"  Estimated Edge: {edge:.1f}pp")
    print()

print("PIPELINE COMPLETE")

INSIDER ALPHA PIPELINE — REAL EXECUTION

INPUT: 1 real prediction market
  Question: Will GPT-4.5 be released by February 28?...
  Volume: $141,504
  Entry Price: $0.82

AGENT A: Insider risk analysis (LLM call)
  Score: 7/10
  Confidence: high

AGENT B: Market behavior analysis (LLM + tools)
  Score: 9/10
  Confidence: high
  Signal Direction: YES (price moving toward YES)

REVISION AGENT: Pattern detection (Deterministic)
  Pattern: DIRECTIONAL_CONFLICT
  Recommendation: GO_EVALUATE
  Explanation: Both agents signal high confidence (A=7, B=9) with B direction=YES. Decision Agent resolves via weighting.

DECISION AGENT: Investment decision (Deterministic, config=aggressive)
  Action: INVEST
  Bet Direction: YES
  Estimated Edge: 12.8pp

PIPELINE COMPLETE


---

## What Just Happened

You ran the full **Insider Alpha pipeline** on a real Polymarket market:

1. **Agent A** scored insider risk from the market description (LLM call, blind to price data)
2. **Agent B** detected price/volume anomalies using deterministic tools + LLM synthesis (blind to market context)
3. **Revision Agent** cross-validated both reports, checking for coherence and known conflict patterns (e.g. `DIRECTIONAL_CONFLICT`, `PUBLIC_INFO_ADJUSTED`)
4. **Decision Agent** synthesized all signals into an INVEST / PASS / WATCH recommendation

The Revision and Decision agents each support two modes — **deterministic** (instant, rule-based) and **LLM** (real API call). Toggle between them in the Interactive Controls section above.

In the full LangGraph pipeline (`src/graph.py`), the Revision Agent can also send feedback back to Agent A or B for up to 2 re-analysis iterations before passing its recommendation forward. This demo runs each agent once.

**Want to try another market?** Move the slider in Section 3 and re-run from there.